In [1]:
import os
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Hyperparameters

The current set of hyperparameters for our model implementation is listed below:

In [2]:
# hyperparameters
batch_size = 32              # how many independent sequences to parallel-process?
max_context_size = 8         # "block_size" ... maximum context length for predictions
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
loss_estimation_iters = 200  # "eval_iters" ... num. iters for estimate_loss

## "Magical" Helper

The boilerplate logic from the first notebook is captured in `helper.py`.

Importing this module addresses the following:

* fetch the tiny Shakespeare text
* encode the data
* define model vocabulary `chars` and variable `vocab_size`
* define functions for `encode` and `decode`
* split into train and validation datasets
* setting `device` (`cuda` or `cpu`)

In [3]:
from helper import *

def get_batch(split):
    # select training or validation dataset
    data = training_data if split == 'train' else validation_data

    # randomly obtain starting indices for a single batch
    ix = torch.randint(len(data) - max_context_size, (batch_size,))

    # stack up sequences from data that start from the starting indices,
    # of length max_context_size; these will be the idx tensor
    # we use in model.forward
    x = torch.stack([data[i:i+max_context_size] for i in ix]).to(device)

    # stack up sequences from data that start from the index following the starting indices,
    # of length max_context_size; these will be the targets tensor
    # we use in model.forward
    y = torch.stack([data[i+1:i+max_context_size+1] for i in ix]).to(device)

    return x,y

----

## A Simple Language Model

In [4]:
# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        # each token directly reads off the logits for the next token
        # from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape

        # idx and targets are both (B,T) tensor of int
        logits = self.token_embedding_table(idx)  # (B,T,C)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B,T) tensor of indices in the current context
        for _ in range(max_new_tokens):
            # get predictions
            logits, loss = self(idx)

            # focus only on the char in the last time position
            logits = logits[:, -1, :]           # becomes (B,C)

            # apply softmax to get the probabilities
            probs = F.softmax(logits, dim=-1)   # (B,C)

            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)

            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)  # (B,T+1)
        return idx

#### Generation from `model`

`model` has not been trained yet, but let's just have a look at the loss and some generated text as well.

In [5]:
torch.manual_seed(1337)

model = BigramLanguageModel(vocab_size).to(device)

xb,yb = get_batch('train')

logits, loss = model(xb, yb)
print(f"logits shape: {logits.shape}")
print(f"loss: {loss.item():.4f}")

# generate from the model
context = torch.zeros((1, 1), dtype=torch.int, device=device)
print(decode(
    model.generate(
        context,
        max_new_tokens=500
    )[0].tolist()
))

logits shape: torch.Size([256, 65])
loss: 4.6485

pYCXxfRkRZd
wc'wfNfT;OLlTEeC K
jxqPToTb?bXAUG:C-SGJO-33SM:C?YI3a
hs:LVXJFhXeNuwqhObxZ.tSVrddXlaSZaNevjw3cHPyZWk,f'qZa-oizCjmuX
YoR&$FMVTfXibIcB!!BA!$W:CdYlHxcbegRirYeYERnkciK;lxWvHFliqmoGSKtSV&BLqWk -.SGFW.byWjbO!UelIljnF$UV&v.C-hsE3SPyckzby:CUup;MpJssX3Qwty;vJlvBPUuIkyBf&pxY-ggCIgj$k:CGlIkJdlyltSPkqmNaW-wNAXQbjxCevib3sr'T:C-&dE$HZvETERSBfxJ$Fstp-LK3:CJ-xTrg
wALkOdmnubruf?qA skz;3QQkhWTm:CEtxjep$vUMUE$EwffMfMPRrFdXKISKH.JrZKINLIk!a!,iyb&y&a
SadapbWPT:VE!zLtYBTEivVKN.kqfa!a!eyCRrxltpmI&fy;VE?!3MJ


## A Simple Training Loop

In [6]:
%%time

torch.manual_seed(1337)

model = BigramLanguageModel(vocab_size).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(loss_estimation_iters)
        for k in range(loss_estimation_iters):
            X,Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and validation sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss is {losses['train']:.4f}, val loss is {losses['val']:.4f}")

    # sample a batch of training data
    xb,yb = get_batch('train')

    # FORWARD pass
    logits, loss = model(xb, yb)

    # discard gradients left over from previous iteration
    optimizer.zero_grad(set_to_none=True)

    # BACKWARD pass
    loss.backward()

    # OPTIMIZATION:
    # use those gradients to update every trainable parameter
    optimizer.step()

print(f"final loss: {loss.item():.4f}\n")

step 0: train loss is 4.7305, val loss is 4.7241
step 500: train loss is 4.1777, val loss is 4.1811
step 1000: train loss is 3.7309, val loss is 3.7377
step 1500: train loss is 3.3827, val loss is 3.3909
step 2000: train loss is 3.1246, val loss is 3.1278
step 2500: train loss is 2.9429, val loss is 2.9429
step 3000: train loss is 2.8009, val loss is 2.8077
step 3500: train loss is 2.7122, val loss is 2.7127
step 4000: train loss is 2.6468, val loss is 2.6375
step 4500: train loss is 2.5979, val loss is 2.5999
final loss: 2.5868

CPU times: user 6.93 s, sys: 203 ms, total: 7.13 s
Wall time: 7.51 s


### Overfitting

Whenever you see training loss dip below validation loss, this means that the model is becoming so good at predictions on the training data that it now struggles to generalize on inputs from unseen data (validation dataset).

_Regularization deals with this overfitting, and we will address this at a later stage._

#### Generation from `model` _after_ training

After training, the output from the current naive `model` implementation should like slightly less random than earlier before training.

In [7]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.int, device=device)
print(decode(
    model.generate(
        context,
        max_new_tokens=500
    )[0].tolist()
))




CExfikRO:
wcowi,STHOLOLETHAKEY: set bobe d e.
S:gO:33SA:


LTanhe:
WanthaiNusqhe, vet? cedXENDoate awice my.

Thstacomzoroup
Yow&$FMOUf isth bot mil;KI!
WARCKI iree sengmin lat Heriliov ts, anend n nghir.
Swanousel lind me l.
MAull ce hiry:
Supr aisspllw y.
Jurinke noroopetelaves
MP:

Pl, d motSSkllo W-S:
FourtCoiib3s the m dourivETENGShire s p-LOK:

PxTre

ALONomnterupt f s ar iris! m:

Thiny aleronth,
MadPre?d my o myr f-NLIE!
Ktied&y, wardsal thisE:zLAnd uin cNI ayaraney Iry ts I&fr t c!
My


----

## Count the Model Weights

How many parameters does this model implementation have?

In [8]:
c = 0
for name, p in model.named_parameters():
    print(f"{name:60s} {p.numel():>10,}")
    c += p.numel()

print(f"{''.join(['-']*71)}")
print(f"{'total parameters':60s} {c:>10,}")

token_embedding_table.weight                                      4,225
-----------------------------------------------------------------------
total parameters                                                  4,225
